### Cell 1 Configuration

In [ ]:
import os
import torch

class Config:
    # ... other settings
    CSV_TRAIN = "train.csv"
    # Make sure this matches your folder name EXACTLY (case-sensitive!)
    IMG_DIR_TRAIN = "data/train"
    # Model Settings
    IMAGE_SIZE = 224
    BATCH_SIZE = 32
    NUM_WORKERS = 4  
    
    # Hyperparameters
    EPOCHS = 6
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-2
    VAL_SPLIT_RATIO = 0.2
    RANDOM_SEED = 42
    
    # Hardware Device Detection
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using Device: {Config.DEVICE}")
if Config.DEVICE.type == 'cuda':
    print(f"🔥 GPU Model: {torch.cuda.get_device_name(0)}")

Using Device: cuda
🔥 GPU Model: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [22]:
# =====================================================================
# CELL 2: DATASET & DATALOADERS
# =====================================================================
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# 1. Import your custom class and transforms from your new Python file
from dataset import IlluminationDataset, train_transforms, val_transforms

# 2. Stratified Splitting
df_metadata = pd.read_csv(Config.CSV_TRAIN)
label_col_name = 'label' if 'label' in df_metadata.columns else 'Label'

train_df, val_df = train_test_split(
    df_metadata, 
    test_size=Config.VAL_SPLIT_RATIO, 
    stratify=df_metadata[label_col_name], 
    random_state=Config.RANDOM_SEED
)

# 3. Instantiate PyTorch Datasets
train_dataset = IlluminationDataset(train_df, Config.IMG_DIR_TRAIN, transform=train_transforms)
val_dataset = IlluminationDataset(val_df, Config.IMG_DIR_TRAIN, transform=val_transforms)

# 4. Prepare Dataloaders utilizing local GPU optimizations
train_loader = DataLoader(
    train_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=True, 
    num_workers=Config.NUM_WORKERS, 
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=False, 
    num_workers=Config.NUM_WORKERS, 
    pin_memory=True
)

print(f"✅ Data Preparation Complete: {len(train_dataset)} Train samples, {len(val_dataset)} Validation samples.")

✅ Data Preparation Complete: 1200 Train samples, 300 Validation samples.


### CELL 3: MODEL ARCHITECTURE & LAYER FREEZING

In [23]:
from dataset import IlluminationDataset, train_transforms, val_transforms
import torch.nn as nn
import torchvision.models as models

def build_illumination_model():
    # 1. Load the pre-trained ResNet18 model
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)

    # 2. Freeze all layers initially
    for param in model.parameters():
        param.requires_grad = False

    # 3. Unfreeze Layer 4 (The final convolutional block)
    # This allows the network to learn domain-specific illumination features
    for param in model.layer4.parameters():
        param.requires_grad = True

    # 4. Replace the Fully Connected (fc) classification head
    # Newly created layers automatically have requires_grad = True
    num_features = model.fc.in_features
    
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),            # Regularization to prevent overfitting
        nn.Linear(num_features, 3)    # 3 Output nodes: Dark(0), Normal(1), Bright(2)
    )

    return model

# Instantiate the model and move it to the GPU/CPU
model = build_illumination_model().to(Config.DEVICE)

# Verify the freezing strategy worked by counting parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"✅ Model Initialized on {Config.DEVICE}!")
print(f"🔒 Total Parameters: {total_params:,}")
print(f"🔥 Trainable Parameters: {trainable_params:,} (Layer 4 + FC Head)")

✅ Model Initialized on cuda!
🔒 Total Parameters: 11,178,051
🔥 Trainable Parameters: 8,395,267 (Layer 4 + FC Head)


#### Training on 8 million parameters

### CELL 4: TRAINING & VALIDATION ENGINE

In [24]:
import time
from tqdm import tqdm # Great for visual progress bars in VS Code

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0
    
    # Progress bar for the terminal/notebook
    loop = tqdm(dataloader, leave=False, desc="Training")
    
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)
        
        # 1. Forward Pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # 2. Backward Pass & Optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 3. Track Metrics
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1) # Get the index of the highest probability
        correct_preds += torch.sum(preds == labels.data).item()
        total_samples += labels.size(0)
        
        # Update progress bar
        loop.set_postfix(loss=loss.item())
        
    epoch_loss = running_loss / total_samples
    epoch_acc = correct_preds / total_samples
    return epoch_loss, epoch_acc

def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0
    
    with torch.no_grad(): # Disable gradient tracking for speed and memory
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct_preds += torch.sum(preds == labels.data).item()
            total_samples += labels.size(0)
            
    epoch_loss = running_loss / total_samples
    epoch_acc = correct_preds / total_samples
    return epoch_loss, epoch_acc

### CELL 5: MAIN EXECUTION LOOP

In [25]:
# 1. Define Loss and Optimizer
criterion = nn.CrossEntropyLoss()

# ONLY pass the trainable parameters to the optimizer
trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer = torch.optim.AdamW(
    trainable_params, 
    lr=Config.LEARNING_RATE, 
    weight_decay=Config.WEIGHT_DECAY
)

# 2. Initialize Tracking Variables
best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("🔥 Starting Training...")
start_time = time.time()

# 3. The Main Epoch Loop
for epoch in range(Config.EPOCHS):
    print(f"\nEpoch {epoch+1}/{Config.EPOCHS}")
    print("-" * 20)
    
    # Train and Validate
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, Config.DEVICE)
    val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, Config.DEVICE)
    
    # Log metrics
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    
    # 4. Checkpointing Strategy (Save ONLY if Accuracy improves)
    if val_acc > best_val_acc:
        print(f"⭐ Validation Accuracy improved from {best_val_acc:.4f} to {val_acc:.4f}. Saving checkpoint!")
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')

time_elapsed = time.time() - start_time
print(f"\n✅ Training Complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
print(f"🏆 Best Validation Accuracy: {best_val_acc:.4f}")

🔥 Starting Training...

Epoch 1/6
--------------------


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/vaibhav_09/Predicting Illumination from Camera Images/myvenv/lib/python3.14/site-packages/torch/utils/data/_utils/worker.py", line 374, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/home/vaibhav_09/Predicting Illumination from Camera Images/myvenv/lib/python3.14/site-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/home/vaibhav_09/Predicting Illumination from Camera Images/dataset.py", line 40, in __getitem__
    image = Image.open(img_path).convert('RGB')
            ~~~~~~~~~~^^^^^^^^^^
  File "/home/vaibhav_09/Predicting Illumination from Camera Images/myvenv/lib/python3.14/site-packages/PIL/Image.py", line 3639, in open
    fp = builtins.open(filename, "rb")
FileNotFoundError: [Errno 2] No